# ME 415 — Homework 1 starter notebook

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/byuflowlab/flightlab/blob/main/notebooks/hw1_starter.ipynb)

## Atmosphere and parasitic drag

This notebook supports the Python portions of HW1. You will still build and inspect the canonical RC-1 aircraft in the FlightLab Workbench. The project file downloaded from the workbench becomes the input to this notebook.

Replace every **TODO** and every `np.nan` placeholder. Explain your results in the response cells; a graph or number without interpretation is not a complete engineering answer.

## Workflow

1. Complete Problem 1a in the FlightLab Workbench.
2. Download the project as `rc1-hw1.flightlab.json`.
3. If you are using Colab, run the cells below in order and upload that file when prompted.
4. If you are working locally, put the project file in the same folder as this notebook.
5. Save or download your completed notebook frequently. A Colab runtime is temporary.

## 0. Set up FlightLab

Run the next cell once whenever you start a new Colab runtime. It checks for the current tested course build and installs it in Colab's temporary Python environment. The GitHub archive is temporary course infrastructure and will be replaced by a shorter PyPI installation after FlightLab is published there.

In [ ]:
import re
import subprocess
import sys
from urllib.request import urlopen

_fallback_commit = "0ee06b60ba2d657cb7dbe324faef81d2c8be8e5a"
_release_url = (
    "https://raw.githubusercontent.com/byuflowlab/flightlab/"
    "main/student_setup/release.txt"
)
try:
    _course_commit = urlopen(_release_url, timeout=10).read().decode().strip()
    if re.fullmatch(r"[0-9a-f]{40}", _course_commit) is None:
        raise ValueError("invalid course release")
except (OSError, UnicodeError, ValueError):
    _course_commit = _fallback_commit
    print("Update check unavailable; using the notebook's included course build.")

_requirement = (
    "flightlab @ https://github.com/byuflowlab/flightlab/archive/"
    f"{_course_commit}.zip"
)
print(f"Installing FlightLab course build {_course_commit[:8]}...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", _requirement])

In [ ]:
from copy import deepcopy
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import brentq

import flightlab
from flightlab import atmos
from flightlab.project import AircraftProject
from flightlab.project_analysis import run_design_point

print(f"FlightLab {flightlab.__version__} is ready.")

## 1. Load your workbench project

The following cell opens an upload chooser in Colab. In a local notebook it looks for `rc1-hw1.flightlab.json` beside the notebook. The uploaded file is your model—not an answer file supplied with the notebook.

In [ ]:
try:
    from google.colab import files
except ImportError:
    files = None

if files is not None:
    uploaded = files.upload()
    project_files = [name for name in uploaded if name.endswith(".flightlab.json")]
    if len(project_files) != 1:
        raise ValueError("Upload exactly one .flightlab.json project file.")
    project_filename = project_files[0]
    project = AircraftProject.from_json(uploaded[project_filename].decode("utf-8"))
else:
    project_filename = "rc1-hw1.flightlab.json"
    project_path = Path(project_filename)
    if not project_path.exists():
        raise FileNotFoundError(
            f"Put {project_filename} in the notebook's current folder, then rerun this cell."
        )
    project = AircraftProject.load(project_path)

print(f"Loaded {project_filename}: {project.name}")

In [ ]:
# These checks only repeat dimensions and names stated in the assignment.
project.require_valid()
wing = project.surface_named("Main wing")
if wing is None:
    raise ValueError("The project must contain a surface named 'Main wing'.")

S_ref, b_ref, c_ref = project.reference_quantities()
assert np.isclose(S_ref, 0.192), f"Check the main-wing geometry: Sref={S_ref:g} m^2"
assert np.isclose(b_ref, 1.200), f"Check the main-wing geometry: bref={b_ref:g} m"
assert np.isclose(c_ref, 0.160), f"Check the main-wing geometry: cref={c_ref:g} m"
assert {"Launch", "Cruise", "Fast"}.issubset(case.name for case in project.cases)
assert project.body_named("Fuselage pod") is not None

print("Basic project checks passed.")
print(f"Sref={S_ref:.3f} m^2, span={b_ref:.3f} m, chord={c_ref:.3f} m, AR={b_ref**2/S_ref:.2f}")

## Problem 1a — Report the workbench results

Enter the three trimmed design-point results you obtained in the workbench. Preserve enough digits to make later comparisons meaningful.

In [ ]:
# TODO: replace np.nan with your workbench results.
workbench_results = {
    "Launch": {"alpha_deg": np.nan, "CL": np.nan, "elevator_deg": np.nan, "L_over_D": np.nan},
    "Cruise": {"alpha_deg": np.nan, "CL": np.nan, "elevator_deg": np.nan, "L_over_D": np.nan},
    "Fast": {"alpha_deg": np.nan, "CL": np.nan, "elevator_deg": np.nan, "L_over_D": np.nan},
}

print(f"{'case':<9} {'alpha [deg]':>12} {'CL':>10} {'elevator [deg]':>16} {'L/D':>10}")
for case_name, row in workbench_results.items():
    print(
        f"{case_name:<9} {row['alpha_deg']:>12.3f} {row['CL']:>10.4f} "
        f"{row['elevator_deg']:>16.3f} {row['L_over_D']:>10.2f}"
    )

**Parasitic-drag observation:** Replace this text with a short comparison of the component contributions shown by the workbench. Identify the largest contributors and explain whether the result is physically reasonable.

## Problem 1b — Constrained pod-shape study

For every trial pod length, preserve both the baseline volume and the width-to-height ratio. Make a fresh copy of the baseline project for every trial so that changes do not accumulate.

Starting from $V=Lwh$ and $r=w/h$, derive expressions for $h(L)$ and $w(L)$ on paper before filling in the two marked lines.

In [ ]:
baseline = deepcopy(project)
volume = 0.400 * 0.110 * 0.100  # m^3
width_to_height = 1.10
lengths = np.linspace(0.250, 0.695, 31)

widths = []
heights = []
drag = []

for length in lengths:
    candidate = deepcopy(baseline)
    pod = candidate.body_named("Fuselage pod")
    if pod is None:
        raise KeyError("Fuselage pod not found")

    # TODO: translate your two constraint equations into Python.
    height = np.nan
    width = np.nan
    if not (np.isfinite(height) and np.isfinite(width)):
        raise NotImplementedError("Replace height and width with your constraint equations.")

    pod.length = float(length)
    pod.width = float(width)
    pod.height = float(height)
    candidate.require_valid()

    result = run_design_point(candidate, candidate.case("Cruise"), ns=28, nc=4)
    widths.append(width)
    heights.append(height)
    drag.append(result.drag)

drag = np.asarray(drag)
print(f"Completed {len(drag)} Cruise design points.")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(lengths, drag, "o-", markersize=4)
ax.set_xlabel("pod length [m]")
ax.set_ylabel("total aircraft drag [N]")
ax.set_title("RC-1 constrained pod-shape study — Cruise")
ax.grid(True, alpha=0.3)
plt.show()

**Pod-study interpretation:** State the baseline drag, the best sampled length and drag, and the percent change from the baseline. Describe the trend rather than only reporting the minimum. Also identify the geometric or packaging reason the sweep stops at 0.695 m.

## Problem 2a — RC aircraft versus transport aircraft

Use the standard-atmosphere states and the relationships

$$M=V/a, \qquad Re_c=\rho Vc/\mu, \qquad C_L=\frac{mg}{(\rho V^2/2)S}. $$

The RC speed is already TAS. Convert the transport's Mach number to TAS using the local speed of sound.

In [ ]:
g = 9.80665  # m/s^2

rc = {"mass": 0.750, "altitude": 1400.0, "speed": 12.0, "area": 0.192, "chord": 0.160}
transport = {"mass": 230_000.0, "altitude": 10_000.0, "mach": 0.85, "area": 377.0, "chord": 6.27}

air_rc = atmos.at(rc["altitude"])
air_transport = atmos.at(transport["altitude"])

# TODO: calculate the transport TAS, then Mach, Reynolds number, and CL for each aircraft.
transport_speed = np.nan
mach_rc = np.nan
re_rc = np.nan
cl_rc = np.nan
re_transport = np.nan
cl_transport = np.nan

print(f"{'quantity':<24} {'RC-1':>15} {'transport':>15}")
print(f"{'Mach number':<24} {mach_rc:>15.5f} {transport['mach']:>15.5f}")
print(f"{'chord Reynolds number':<24} {re_rc:>15.5e} {re_transport:>15.5e}")
print(f"{'required lift coefficient':<24} {cl_rc:>15.5f} {cl_transport:>15.5f}")

**Comparison:** Which dimensionless operating parameter is surprisingly similar? Which parameters differ by orders of magnitude? Briefly explain what those differences mean physically.

## Problem 2b — Why does a transport step-climb?

First keep altitude and Mach fixed while the mass falls to 170,000 kg. Then find the altitude of an ideal schedule that keeps both Mach and the beginning-of-cruise lift coefficient fixed. At fixed Mach, $q=(\gamma/2)pM^2$, so use the lift equation to determine how the required pressure scales with mass.

In [ ]:
end_mass = 170_000.0  # kg

# TODO: required CL after the fuel burn if altitude and Mach remain fixed.
cl_end_fixed_altitude = np.nan

# TODO: pressure required to hold the initial Mach and CL at the lower mass.
target_pressure = np.nan

# Once target_pressure is known, this numerically inverts the standard atmosphere.
if np.isfinite(target_pressure):
    end_altitude = brentq(
        lambda altitude: atmos.pressure(altitude) - target_pressure,
        0.0,
        20_000.0,
    )
else:
    end_altitude = np.nan

print(f"End-of-cruise CL at 10,000 m: {cl_end_fixed_altitude:.5f}")
print(f"Ideal constant-M, constant-CL pressure: {target_pressure:.1f} Pa")
print(f"Ideal end-of-cruise altitude: {end_altitude:.0f} m")

**Step-climb explanation:** Explain why the ideal altitude rises as fuel is burned. Then explain why an actual transport uses discrete step climbs rather than following that altitude continuously. Mention relevant operational or performance constraints, not just the algebra.

## Submission check

Before submitting, confirm that:

- every `np.nan`, **TODO**, and italic response prompt has been replaced;
- the notebook runs in order from a fresh runtime after uploading your project;
- the pod-sweep plot has labeled axes and units;
- numerical results include appropriate units and precision;
- your written discussion answers the physical questions; and
- you have saved or downloaded the completed notebook.